# ROCm 10.2 on Radeon 780M (gfx1103)

Local virtual environment: the ROCm runtime (`rocm-sdk-*`) and PyTorch are installed in `.venv`; system-wide `/opt/rocm` is not used.

The notebook kernel is Python 3.13 from `.venv` (select `.venv/bin/python` in VS Code / Cursor, or run `uv run jupyter lab`).

In [1]:
import os, platform, time
import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__)
print("hip    :", torch.version.hip)
print("cuda   :", torch.cuda.is_available())
print("HSA_OVERRIDE:", os.environ.get("HSA_OVERRIDE_GFX_VERSION", "(not set, correct)"))

assert torch.cuda.is_available(), "GPU not detected: check /dev/kfd, render/video groups, and rocm-sdk-*"
p = torch.cuda.get_device_properties(0)
print("device :", torch.cuda.get_device_name(0))
print("arch   :", p.gcnArchName)
print("memory : %.2f GiB (GTT)" % (p.total_memory / 2**30))
print("CU     :", p.multi_processor_count)

python : 3.13.13
torch  : 2.14.0+rocm10.2.0
hip    : 7.17.26434
cuda   : True
HSA_OVERRIDE: (not set, correct)
device : AMD Radeon 780M Graphics
arch   : gfx1103
memory : 7.44 GiB (GTT)
CU     : 6


## 1. Conv2D benchmark (CPU vs GPU)

Forward pass for a `16x3x128x128` batch with 32 output channels and a `3x3` kernel. CPU uses fp32 and GPU uses fp16; warm-up (including MIOpen setup) is excluded from timings.

In [2]:
import torch.nn.functional as F

def bench_conv2d(device, dtype, batch_size=16, image_size=128, in_channels=3,
                 out_channels=32, kernel_size=3, iters=20, warmup=5):
    x = torch.randn(batch_size, in_channels, image_size, image_size,
                    device=device, dtype=dtype)
    weight = torch.randn(out_channels, in_channels, kernel_size, kernel_size,
                         device=device, dtype=dtype)
    if device == "cuda":
        x = x.to(memory_format=torch.channels_last)
        weight = weight.to(memory_format=torch.channels_last)
        torch.backends.cudnn.benchmark = True
    for _ in range(warmup):
        F.conv2d(x, weight, padding=kernel_size // 2)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        F.conv2d(x, weight, padding=kernel_size // 2)
    if device == "cuda":
        torch.cuda.synchronize()
    sec = (time.perf_counter() - t0) / iters
    flops = 2 * batch_size * image_size**2 * out_channels * in_channels * kernel_size**2
    return sec, flops / sec / 1e9

conv_rows = [
    ("CPU fp32", *bench_conv2d("cpu", torch.float32)),
    ("GPU fp16", *bench_conv2d("cuda", torch.float16)),
]
for name, sec, gflops in conv_rows:
    print(f"{name:<9} {sec * 1e3:8.2f} ms   {gflops:8.1f} GFLOPS")
speedup = conv_rows[1][2] / conv_rows[0][2]
print(f"GPU fp16 speedup vs CPU fp32: x{speedup:.2f}")

MIOpen: Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1103: libMIOpenCKGroupedConv_gfx1103.so: cannot open shared object file: No such file or directory


CPU fp32     11.00 ms       41.2 GFLOPS
GPU fp16      0.75 ms      602.4 GFLOPS
GPU fp16 speedup vs CPU fp32: x14.63


## 2. Transformer operations (Attention, Softmax, backward)

Manual scaled dot-product attention for a batch of 4, 8 heads, 512 tokens, and 64 dimensions per head. The benchmark includes forward and backward passes; CPU uses fp32 and GPU fp16. It checks output shape, finite values and gradients, and Softmax row sums. GFLOPS counts attention matrix multiplications, not Softmax operations.

In [ ]:
def bench_transformer_ops(device, dtype, batch_size=4, seq_len=512,
                          num_heads=8, head_dim=64, iters=10, warmup=2):
    shape = (batch_size, num_heads, seq_len, head_dim)
    q, k, v = (torch.randn(*shape, device=device, dtype=dtype, requires_grad=True)
               for _ in range(3))

    def step():
        q.grad = k.grad = v.grad = None
        scores = torch.matmul(q, k.transpose(-2, -1)) * (head_dim ** -0.5)
        weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(weights, v)
        loss = output.float().square().mean()
        loss.backward()
        return weights, output, loss

    for _ in range(warmup):
        step()
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        weights, output, loss = step()
    if device == "cuda":
        torch.cuda.synchronize()
    sec = (time.perf_counter() - t0) / iters

    expected_shape = (batch_size, num_heads, seq_len, head_dim)
    assert output.shape == expected_shape, f"unexpected output shape: {output.shape}"
    assert torch.isfinite(output).all().item() and torch.isfinite(weights).all().item()
    row_sums = weights.float().sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-2, rtol=1e-2)
    grads = (q.grad, k.grad, v.grad)
    assert all(g is not None and torch.isfinite(g).all().item() for g in grads)
    grad_magnitude = sum(g.float().abs().sum() for g in grads if g is not None)
    assert grad_magnitude.item() > 0

    flops = 12 * batch_size * num_heads * seq_len**2 * head_dim
    return sec, flops / sec / 1e9, float(loss.item())

transformer_rows = [
    ("CPU fp32", *bench_transformer_ops("cpu", torch.float32)),
    ("GPU fp16", *bench_transformer_ops("cuda", torch.float16)),
]
for name, sec, gflops, loss in transformer_rows:
    print(f"{name:<9} {sec * 1e3:8.2f} ms   {gflops:8.1f} GFLOPS  loss={loss:.5f} checks=ok")
speedup = transformer_rows[1][2] / transformer_rows[0][2]
print(f"GPU fp16 speedup vs CPU fp32: x{speedup:.2f}")

## 3. Sanity check and matmul benchmark

`4096x4096` in fp16/bf16. APUs are often limited by shared DDR5 memory rather than peak TFLOPS, so results may be lower than on a discrete GPU.

In [3]:
def bench_matmul(device, dtype, n=4096, iters=15, warmup=5):
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    for _ in range(warmup):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / iters
    return dt, 2 * n**3 / dt / 1e9

for name, dev, dtype in [
    ("CPU fp32", "cpu", torch.float32),
    ("GPU fp16", "cuda", torch.float16),
    ("GPU bf16", "cuda", torch.bfloat16),
]:
    sec, gflops = bench_matmul(dev, dtype)
    print(f"{name:<9} {sec * 1e3:8.2f} ms   {gflops:8.1f} GFLOPS")

CPU fp32    201.82 ms      681.0 GFLOPS
GPU fp16     23.26 ms     5910.1 GFLOPS
GPU bf16     23.13 ms     5942.7 GFLOPS


## 4. Small training run: CPU vs iGPU

Synthetic 32x32 images (each class has its own prototype plus noise), a small CNN, and AMP on the GPU.

**The first** GPU convolution call triggers MIOpen autotuning/JIT and may be noticeably slower; the function below excludes it from the measurement.

In [4]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(128, 10)

    def forward(self, x):
        return self.head(self.features(x).flatten(1))

g = torch.Generator().manual_seed(0)
proto = torch.randn(10, 3, 8, 8, generator=g)
ytr = torch.randint(0, 10, (4096,), generator=g)
yte = torch.randint(0, 10, (1024,), generator=g)
xtr = F.interpolate(proto[ytr], size=32, mode="bilinear") + 0.6 * torch.randn(4096, 3, 32, 32, generator=g)
xte = F.interpolate(proto[yte], size=32, mode="bilinear") + 0.6 * torch.randn(1024, 3, 32, 32, generator=g)
print(xtr.shape, ytr.shape, xte.shape, yte.shape)

torch.Size([4096, 3, 32, 32]) torch.Size([4096]) torch.Size([1024, 3, 32, 32]) torch.Size([1024])


In [5]:
import copy

def train(device, epochs=2, bs=64):
    torch.manual_seed(42)
    model = SmallCNN().to(device)
    if device == "cuda":
        model = model.to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
    loss_fn = nn.CrossEntropyLoss()
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    dl = DataLoader(TensorDataset(xtr, ytr), batch_size=bs, shuffle=True)

    model.train()
    if device == "cuda":
        # Exclude MIOpen autotuning/JIT from the measurement.
        torch.backends.cudnn.benchmark = True
        warm = copy.deepcopy(model)
        xb, yb = next(iter(dl))
        xb = xb.to(device).to(memory_format=torch.channels_last)
        yb = yb.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            wl = loss_fn(warm(xb), yb)
        scaler.scale(wl).backward()
        torch.cuda.synchronize()
        del warm
        torch.cuda.empty_cache()

    t0 = time.perf_counter()
    for _ in range(epochs):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            if device == "cuda":
                xb = xb.to(memory_format=torch.channels_last)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
    if device == "cuda":
        torch.cuda.synchronize()
    sec = time.perf_counter() - t0

    model.eval()
    xb, yb = xte.to(device), yte.to(device)
    if device == "cuda":
        xb = xb.to(memory_format=torch.channels_last)
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
        acc = (model(xb).argmax(1) == yb).float().mean().item()
    return sec, float(loss), acc

for dev in ["cpu", "cuda"]:
    sec, loss, acc = train(dev)
    print(f"{dev:>4}: {sec:6.2f}s   last loss={loss:.3f}   test acc={acc * 100:.1f}%")

/tmp/ipykernel_78845/3981522963.py:51: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/rockrel/rockrel/external-builds/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  return sec, float(loss), acc


 cpu:   2.46s   last loss=0.006   test acc=100.0%
cuda:   0.84s   last loss=0.006   test acc=100.0%


## 5. Memory and important notes

- GPU memory is GTT, shared with system RAM. Before training, consider stopping Ollama (`sudo systemctl stop ollama`) so the processes do not compete for the available memory.
- `HSA_OVERRIDE_GFX_VERSION` is not needed for `gfx1103` and may cause problems.
- The first convolution call may show a MIOpen warning about a missing `libMIOpenCKGroupedConv_gfx1103.so`. This is not necessarily an error; some algorithms are simply unavailable.
- To increase the available memory budget, adjust `amdgpu.gttsize` in the kernel parameters (a system-level setting not changed by this notebook).

In [6]:
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free/total: {free / 2**30:.2f} / {total / 2**30:.2f} GiB")
print(f"peak allocated: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")
print(f"reserved:       {torch.cuda.memory_reserved() / 2**30:.2f} GiB")

GPU memory free/total: 6.86 / 7.44 GiB
peak allocated: 0.22 GiB
reserved:       0.16 GiB
